# Graph RAG — Method 1 (Cypher-first → Vector fallback) — Batch over 35 questions

**Pipeline (Method 1, unchanged):**
1. LLM generates Cypher from the schema + question.
2. Run the Cypher against Neo4j.
3. If the graph returns useful rows → answer from the graph.
4. Otherwise → vector retrieval over the graph text → answer from retrieved context.

This version fixes reproducibility (runs top-to-bottom), embeds real definition text
(`rdfs__comment`) instead of bare labels, never loses a question on a stage error, and
carries the dataset `id` through to the output so a separate scoring notebook can join on it.

**Prepare a `.env` with:** `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `OPENAI_API_KEY`.


In [1]:
import os
import json
import time
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI            # LLM for Cypher + answering (kept)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Embeddings: use the SAME model as VRAG (Ollama nomic-embed-text), NOT OpenAI.
# pip install langchain-ollama   (run once if not installed)
try:
    from langchain_ollama import OllamaEmbeddings
except ImportError:
    from langchain_community.embeddings import OllamaEmbeddings

# Prefer the maintained langchain_neo4j package; fall back to community if needed.
try:
    from langchain_neo4j import Neo4jGraph, Neo4jVector
except ImportError:
    from langchain_community.graphs import Neo4jGraph
    from langchain_neo4j import Neo4jVector


c:\Users\kyith\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.prompts import PromptTemplate

## 1) Load environment variables

In [3]:
load_dotenv()

required_env = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "OPENAI_API_KEY"]
missing = [k for k in required_env if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}")

NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


## 2) Connect to Neo4j and refresh schema

In [4]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
)

graph.refresh_schema()
schema = graph.schema
print(schema[:1500])  # sanity-check what the LLM will see


Node properties:
Section {embedding: LIST, text: STRING, in_scope: BOOLEAN, has_text: BOOLEAN, number: INTEGER, title: STRING, chapter: STRING, part: STRING}
Concept {nomic_embedding: LIST, name: STRING, embedding: LIST, nomic_text: STRING, text: STRING, label: STRING, definition: STRING}
LegalAct {name: STRING}
Chapter {name: STRING, title: STRING}
Part {name: STRING, title: STRING}
Exemption {name: STRING, comment: STRING, display: STRING, ref: STRING}
ConsentException {name: STRING, comment: STRING, ref: STRING, display: STRING}
LawfulBasis {name: STRING, ref: STRING, display: STRING, comment: STRING}
Relationship properties:
REQUIRES {predicate: STRING}
RELATES {predicate: STRING, section: INTEGER}
The relationships:
(:Concept)-[:SUBCLASS_OF]->(:Concept)
(:Concept)-[:CITED_IN]->(:Section)
(:Concept)-[:DEFINED_IN]->(:Section)
(:Concept)-[:REQUIRES]->(:Concept)
(:Concept)-[:RELATES]->(:Concept)
(:LegalAct)-[:HAS_CHAPTER]->(:Chapter)
(:Chapter)-[:HAS_PART]->(:Part)
(:Chapter)-[:HAS_SE

## 3) LLM

In [5]:
llm_4omini = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0,
)


## 4) Vector store (fallback) — uses nomic, the SAME embedding as VRAG

This reads the `rdfs__comment` text **already in your graph** and makes fresh nomic
fingerprints of it, stored in a NEW index (`concept_nomic_index`). It does NOT touch your
build script and does NOT replace your existing mpnet vectors — those just stay unused.

`from_existing_graph` only embeds nodes that don't yet have the nomic property, so this cell
is safe to re-run (the first run does the embedding, later runs are instant).

Requires Ollama running locally with the model pulled:  `ollama pull nomic-embed-text`


In [6]:
# --- config ---
NODE_LABEL      = "Concept"                 # domain nodes for the fallback
INDEX_NAME      = "concept_nomic_index"     # NEW index — separate from the mpnet 'embedding' index
EMBED_PROPERTY  = "nomic_embedding"         # NEW property — separate from mpnet 'embedding'
DISPLAY_PROP    = "nomic_text"              # stores the exact text that was embedded (text + definition)
RETRIEVER_K     = 4
# --------------
import time as _t

# Same embedding model as VRAG.
embedder = OllamaEmbeddings(model="nomic-embed-text")
# embedder = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

# 0) Confirm Ollama actually returns vectors (errors loudly here if it is not running)
probe_vec = embedder.embed_query("personal data")
DIM = len(probe_vec)
print(f"Ollama OK — embedding dimension = {DIM}")

# 1) CLEAN SLATE: remove any broken index + stale properties from earlier attempts
graph.query(f"DROP INDEX {INDEX_NAME} IF EXISTS")
graph.query(f"MATCH (n:{NODE_LABEL}) REMOVE n.{EMBED_PROPERTY}, n.{DISPLAY_PROP}")
print("Cleared old index and stale properties.")

# 2) Pull Concept nodes + the text to embed (text + definition)
rows = graph.query(f'''
MATCH (n:{NODE_LABEL})
WHERE n.text IS NOT NULL OR n.definition IS NOT NULL
RETURN elementId(n) AS eid,
       trim(coalesce(n.text,'') + ' ' + coalesce(n.definition,'')) AS text
''')
rows = [r for r in rows if r["text"].strip()]
print(f"Nodes to embed: {len(rows)}")
if len(rows) == 0:
    raise ValueError("No Concept nodes had text/definition — check NODE_LABEL and properties.")

# 3) Embed (text + definition) with nomic; write the vector AND the embedded string
texts   = [r["text"] for r in rows]
vectors = embedder.embed_documents(texts)
graph.query(
    f"UNWIND $batch AS row MATCH (n) WHERE elementId(n) = row.eid "
    f"SET n.{EMBED_PROPERTY} = row.vec, n.{DISPLAY_PROP} = row.txt",
    params={"batch": [{"eid": r["eid"], "vec": v, "txt": t}
                      for r, v, t in zip(rows, vectors, texts)]},
)
written = graph.query(f"MATCH (n:{NODE_LABEL}) WHERE n.{EMBED_PROPERTY} IS NOT NULL RETURN count(n) AS c")[0]["c"]
print(f"Vectors written to {written} nodes.")

# 4) Create a FRESH vector index (cosine, matching dimension)
graph.query(f'''
CREATE VECTOR INDEX {INDEX_NAME}
FOR (n:{NODE_LABEL}) ON (n.{EMBED_PROPERTY})
OPTIONS {{ indexConfig: {{
  `vector.dimensions`: {DIM},
  `vector.similarity_function`: 'cosine'
}} }}
''')

# 5) Poll until the index is ONLINE and 100% populated (instead of a blind sleep)
state = None
for _ in range(30):
    idxs = graph.query("SHOW VECTOR INDEXES YIELD name, state, populationPercent "
                       "RETURN name, state, populationPercent")
    hit = [r for r in idxs if r["name"] == INDEX_NAME]
    if hit:
        state = hit[0]
        if state["state"] == "ONLINE" and state["populationPercent"] == 100.0:
            break
    _t.sleep(1)
print(f"Index status: {state}")

# 6) DIRECT index test (bypasses LangChain) — proves whether the index itself works
direct = graph.query(
    f"CALL db.index.vector.queryNodes('{INDEX_NAME}', 4, $vec) "
    f"YIELD node, score RETURN node.{DISPLAY_PROP} AS text, score",
    params={"vec": probe_vec},
)
print(f"Direct index query returned {len(direct)} rows.")
for d in direct[:2]:
    print("   score", round(d["score"], 3), "|", (d["text"] or "")[:90])

# 7) Attach the LangChain retriever to the index
vector_store = Neo4jVector.from_existing_index(
    embedding=embedder,
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD,
    index_name=INDEX_NAME,
    text_node_property=DISPLAY_PROP,
)
retriever = vector_store.as_retriever(search_kwargs={"k": RETRIEVER_K})
_probe = retriever.invoke("What is personal data?")
print(f"Retriever self-check returned {len(_probe)} docs.")
print("Vector store ready (nomic-embed-text over Concept: text + definition).")


Ollama OK — embedding dimension = 768
Cleared old index and stale properties.
Nodes to embed: 73
Vectors written to 73 nodes.
Index status: {'name': 'concept_nomic_index', 'state': 'ONLINE', 'populationPercent': 100.0}
Direct index query returned 4 rows.
   score 0.896 | Definition: "Personal Data" means any information relating to a Person, which enables the 
   score 0.89 | “Personal Data” means any information relating to a Person, which enables the identificati
Retriever self-check returned 4 docs.
Vector store ready (nomic-embed-text over Concept: text + definition).


In [7]:
docs = retriever.invoke("What is personal data?")
print("num docs:", len(docs))
for d in docs:
    print("---")
    print(d.page_content[:200])

num docs: 4
---
Definition: "Personal Data" means any information relating to a Person, which enables the identification of such Person, whether directly or indirectly, but not including the information of the deceas
---
“Personal Data” means any information relating to a Person, which enables the identification of such Person, whether directly or indirectly, but not including the information of the deceased Persons i
---
Any information in any form, including both personal and non-personal data. Any information in any form, including both personal and non-personal data.
---
This refers to the act of gathering or acquiring Personal Data.

The act of obtaining personal data from a data subject or a third party. Collection must be for a notified purpose and limited to what’


## 5) Prompts

Call Prompt Template -- .py

In [8]:
# import importlib.metadata as m
# for p in ["langchain-core", "langchain-openai", "langchain", "langchain-community"]:
#     try:
#         print(p, m.version(p))
#     except m.PackageNotFoundError:
#         print(p, "NOT installed")
from langchain_core.prompts import PromptTemplate
print(PromptTemplate)   # <class 'langchain_core.prompts.prompt.PromptTemplate'>

<class 'langchain_core.prompts.prompt.PromptTemplate'>


In [1]:
import importlib, sys, os
from langchain_core.prompts import PromptTemplate   # same cell as the construction

PROMPT_DIR    = "promptTemplate"
PROMPT_MODULE = "prompt4"

prompt_path = os.path.abspath(PROMPT_DIR)
if prompt_path not in sys.path:
    sys.path.insert(0, prompt_path)

pmod = importlib.import_module(PROMPT_MODULE)
importlib.reload(pmod)
P, PROMPT_VERSION = pmod.PROMPTS, pmod.VERSION

# Raw template strings — required by the plain-replacement Cypher chain.
# The Cypher prompt has literal braces in its examples (e.g. (:Section {number: 24})),
# so it must NOT go through PromptTemplate f-string formatting; _build_cypher_prompt
# below calls .replace() on these raw strings instead.
CYPHER_GENERATION_TEMPLATE = P["cypher_generation"]
QA_TEMPLATE                = P["graph_qa"]
VECTOR_QA_TEMPLATE         = P["vector_qa"]

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template=P["cypher_generation"],
)
QA_PROMPT = PromptTemplate(
    input_variables=["question", "context"],
    template=P["graph_qa"],
)
VECTOR_QA_PROMPT = PromptTemplate(
    input_variables=["question", "retrieved_context"],
    template=P["vector_qa"],
)
print("Loaded:", PROMPT_VERSION, "| PromptTemplate ready")

c:\Users\kyith\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: prompt4 | PromptTemplate ready


In [10]:
cypher_generation_chain = CYPHER_GENERATION_PROMPT | llm_4omini | StrOutputParser()
graph_answer_chain       = QA_PROMPT            | llm_4omini | StrOutputParser()
vector_answer_chain      = VECTOR_QA_PROMPT     | llm_4omini | StrOutputParser()


In [11]:
from langchain_core.runnables import RunnableLambda

# Fill {schema}/{question} by plain replacement so the literal braces in the Cypher
# examples (e.g. {number: 24}) are NOT interpreted as template variables.
def _build_cypher_prompt(inputs: dict) -> str:
    return (CYPHER_GENERATION_TEMPLATE
            .replace("{schema}", inputs["schema"])
            .replace("{question}", inputs["question"]))

cypher_generation_chain = RunnableLambda(_build_cypher_prompt) | llm_4omini | StrOutputParser()
graph_answer_chain       = QA_PROMPT        | llm_4omini | StrOutputParser()
vector_answer_chain      = VECTOR_QA_PROMPT | llm_4omini | StrOutputParser()


## 6) Helpers

In [12]:
def clean_cypher(text: str) -> str:
    """Strip markdown fences and stray prefixes the LLM sometimes adds."""
    if not text:
        return ""
    t = text.replace("```cypher", "").replace("```", "").strip()
    # Drop a leading "Cypher:" / "cypher" line if present
    lines = t.splitlines()
    if lines and lines[0].strip().lower().rstrip(":") == "cypher":
        lines = lines[1:]
    return "\n".join(lines).strip()


def drop_empty_rows(rows):
    """Remove rows whose every value is null/empty so they can't fake a graph hit."""
    cleaned = []
    for row in rows or []:
        if isinstance(row, dict):
            if any(v not in (None, "", [], {}) for v in row.values()):
                cleaned.append(row)
        elif row not in (None, "", [], {}):
            cleaned.append(row)
    return cleaned


def has_useful_graph_result(rows) -> bool:
    return len(drop_empty_rows(rows)) > 0


def format_docs(docs) -> str:
    parts = []
    for i, d in enumerate(docs, start=1):
        parts.append(
            f"[Doc {i}]\n"
            f"Content:\n{d.page_content}\n\n"
            f"Metadata:\n{json.dumps(d.metadata, ensure_ascii=False)}"
        )
    return "\n\n".join(parts)


## 7) Method 1 pipeline

Each stage is wrapped so a failure in Cypher generation, Cypher execution, or vector
fallback still returns a complete row (with `cypher_error` / `error` populated) instead of
aborting the question.


In [13]:
def run_cypher_then_vector(question: str) -> dict:
    out = {
        "mode": None,
        "generated_cypher": None,
        "graph_rows": [],
        "retrieved_context": None,
        "answer": None,
        "cypher_error": None,
        "error": None,
    }

    # Step 1: generate Cypher
    try:
        raw_cypher = cypher_generation_chain.invoke({"schema": schema, "question": question})
        out["generated_cypher"] = clean_cypher(raw_cypher)
    except Exception as e:
        out["error"] = f"cypher_generation_failed: {e}"

    # Step 2: run Cypher
    rows = []
    if out["generated_cypher"]:
        try:
            rows = graph.query(out["generated_cypher"])
        except Exception as e:
            out["cypher_error"] = str(e)
    out["graph_rows"] = rows

    # Step 3: answer from graph if useful
    if has_useful_graph_result(rows):
        try:
            clean_rows = drop_empty_rows(rows)
            out["answer"] = graph_answer_chain.invoke({
                "question": question,
                "context": json.dumps(clean_rows, ensure_ascii=False),
            })
            out["mode"] = "graph"
            return out
        except Exception as e:
            out["error"] = (out["error"] or "") + f" | graph_answer_failed: {e}"

    # Step 4: vector fallback
    try:
        docs = retriever.invoke(question)
        out["retrieved_context"] = format_docs(docs)
        out["answer"] = vector_answer_chain.invoke({
            "question": question,
            "retrieved_context": out["retrieved_context"],
        })
        out["mode"] = "vector_fallback"
    except Exception as e:
        out["error"] = (out["error"] or "") + f" | vector_fallback_failed: {e}"
        out["mode"] = "failed"
        if out["answer"] is None:
            out["answer"] = "I don't know based on the graph."

    return out


### Smoke test (a few questions)

In [14]:
for q in [
    "What is PDPA?",
    "Who are the individuals involved in personal data?",
    "In what cases can personal data be collected, used or disclosed?",
]:
    r = run_cypher_then_vector(q)
    print("Q:", q)
    print("  MODE:", r["mode"])
    print("  CYPHER:", (r["generated_cypher"] or "").replace("\n", " ")[:120])
    print("  CYPHER_ERROR:", r["cypher_error"])
    print("  ERROR:", r["error"])
    print("  ANSWER:", (r["answer"] or "")[:160])
    print()


Q: What is PDPA?
  MODE: graph
  CYPHER: MATCH (c:Concept) WHERE toLower(replace(c.label,' ','')) CONTAINS 'pdpa'    OR toLower(replace(c.name ,' ','')) CONTAINS
  CYPHER_ERROR: None
  ERROR: None
  ANSWER: PDPA stands for the Personal Data Protection Act. It includes various provisions regarding the handling of personal data, such as exemptions where the act does 

Q: Who are the individuals involved in personal data?
  MODE: graph
  CYPHER: MATCH (c:Concept) WHERE toLower(c.label) CONTAINS 'datasubject'    OR toLower(c.label) CONTAINS 'datacontroller'    OR t
  CYPHER_ERROR: None
  ERROR: None
  ANSWER: The individuals involved in personal data include the Data Controller, who is responsible for decisions regarding the collection, use, or disclosure of personal

Q: In what cases can personal data be collected, used or disclosed?
  MODE: graph
  CYPHER: MATCH (lb:LawfulBasis) OPTIONAL MATCH (lb)-[:GROUNDED_IN]->(s:Section) RETURN lb.display, lb.comment, s.number AS sectio
  CYPHER_ERR

## 8) Load the questions

Point `QUESTION_SOURCE` at your full 35-question file. The loader keeps the `id` column so
your separate scoring notebook can join on it. If the file isn't found, it falls back to the
clean 29-question xlsx (in-scope only) and prints a warning.


In [15]:
QUESTION_SOURCE = "../data/pdpa_qa_dataset.xlsx"   # <-- your full 35-question file

def load_questions(path):
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    if "question" not in df.columns:
        raise ValueError(f"No 'question' column in {path}. Columns: {list(df.columns)}")
    df = df.dropna(subset=["question"]).reset_index(drop=True)
    if "id" not in df.columns:
        df["id"] = [f"Q-{i+1:03d}" for i in range(len(df))]
    return df[["id", "question"]]

df_q = load_questions(QUESTION_SOURCE)
print(f"Loaded {len(df_q)} questions.")
# df_q.head()
df_q

Loaded 35 questions.


,id,question
0,PDPA-001,What is PDPA?
1,PDPA-002,What is personal data?
2,PDPA-003,What is a personal data protection policy?
3,PDPA-004,What is a privacy statement?
4,PDPA-005,What if there is no Data Protection Policy?
5,PDPA-006,Who are the individuals involved in personal d...
6,PDPA-007,Who are the personal data controllers and pers...
7,PDPA-008,"In what cases can personal data be collected, ..."
8,PDPA-009,What are the conditions for sending or transfe...
9,PDPA-010,What are the conditions of consent?


## 9) Run the batch

In [16]:
results = []
run_started = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

for i, rec in df_q.iterrows():
    qid, q = rec["id"], rec["question"]
    t0 = time.time()
    row = {
        "id": qid,
        "idx": i + 1,
        "question": q,
        "run_started": run_started,
        "ts": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mode": None,
        "answer": None,
        "generated_cypher": None,
        "graph_rows": None,
        "retrieved_context": None,
        "cypher_error": None,
        "error": None,
        "latency_sec": None,
    }
    try:
        out = run_cypher_then_vector(q)
        row["mode"]              = out["mode"]
        row["answer"]            = out["answer"]
        row["generated_cypher"]  = out["generated_cypher"]
        row["graph_rows"]        = json.dumps(out["graph_rows"], ensure_ascii=False)
        row["retrieved_context"] = out["retrieved_context"]
        row["cypher_error"]      = out["cypher_error"]
        row["error"]             = out["error"]
    except Exception as e:
        row["error"] = f"pipeline_crashed: {e}"
    row["latency_sec"] = round(time.time() - t0, 2)
    results.append(row)
    print(f"[{i+1}/{len(df_q)}] {qid}  mode={row['mode']}  {row['latency_sec']}s")

print("Done.")

[1/35] PDPA-001  mode=graph  4.16s
[2/35] PDPA-002  mode=graph  4.02s
[3/35] PDPA-003  mode=graph  4.3s
[4/35] PDPA-004  mode=vector_fallback  10.73s
[5/35] PDPA-005  mode=graph  3.61s
[6/35] PDPA-006  mode=graph  4.25s
[7/35] PDPA-007  mode=graph  3.43s
[8/35] PDPA-008  mode=graph  6.25s
[9/35] PDPA-009  mode=graph  4.14s
[10/35] PDPA-010  mode=graph  3.17s
[11/35] PDPA-011  mode=vector_fallback  9.43s
[12/35] PDPA-012  mode=vector_fallback  7.12s
[13/35] PDPA-013  mode=graph  2.42s
[14/35] PDPA-014  mode=graph  4.52s
[15/35] PDPA-015  mode=graph  3.54s
[16/35] PDPA-016  mode=graph  2.32s
[17/35] PDPA-017  mode=graph  4.49s
[18/35] PDPA-018  mode=graph  3.61s
[19/35] PDPA-019  mode=graph  3.74s
[20/35] PDPA-020  mode=graph  3.32s
[21/35] PDPA-021  mode=graph  2.57s
[22/35] PDPA-022  mode=graph  6.64s
[23/35] PDPA-023  mode=vector_fallback  7.75s
[24/35] PDPA-024  mode=graph  3.02s
[25/35] PDPA-025  mode=graph  3.48s
[26/35] PDPA-026  mode=graph  3.31s
[27/35] PDPA-027  mode=vector_fal

## 10) Inspect and export

In [17]:
df_out = pd.DataFrame(results)
print(len(df_out), "questions processed.")
print(df_out["mode"].value_counts(dropna=False).to_string())
df_out.head()

35 questions processed.
mode
graph              29
vector_fallback     6


,id,idx,question,run_started,ts,mode,answer,generated_cypher,graph_rows,retrieved_context,cypher_error,error,latency_sec
0,PDPA-001,1,What is PDPA?,2026-06-27 22:45:29,2026-06-27 22:45:29,graph,PDPA stands for the Personal Data Protection A...,MATCH (c:Concept)\nWHERE toLower(replace(c.lab...,"[{""c.label"": ""Exemption"", ""c.definition"": ""Sec...",None,None,None,4.16
1,PDPA-002,2,What is personal data?,2026-06-27 22:45:29,2026-06-27 22:45:33,graph,Personal data refers to information that can i...,MATCH (c:Concept)\nWHERE toLower(replace(c.lab...,"[{""c.label"": ""SensitiveDataProhibition"", ""c.de...",None,None,None,4.02
2,PDPA-003,3,What is a personal data protection policy?,2026-06-27 22:45:29,2026-06-27 22:45:37,graph,A personal data protection policy is an intern...,MATCH (c:Concept)\nWHERE toLower(replace(c.lab...,"[{""c.label"": ""CrossBorderTransfer"", ""c.definit...",None,None,None,4.30
3,PDPA-004,4,What is a privacy statement?,2026-06-27 22:45:29,2026-06-27 22:45:42,vector_fallback,"A privacy statement, also known as a Privacy N...",MATCH (c:Concept)\nWHERE toLower(replace(c.lab...,[],[Doc 1]\nContent:\nPrivacy Notice is a publicl...,None,None,10.73
4,PDPA-005,5,What if there is no Data Protection Policy?,2026-06-27 22:45:29,2026-06-27 22:45:52,graph,"If there is no Data Protection Policy, the Con...",MATCH (c:Concept)\nWHERE toLower(c.label) CONT...,"[{""c.label"": ""PersonalDataProtectionPolicy"", ""...",None,None,None,3.61


In [18]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Prompt version now travels with the file: in the name AND in a dedicated sheet.
xlsx_path = f"graph_rag_method1_results_{PROMPT_VERSION}_{stamp}.xlsx"

# Tag every result row with the prompt version so the scoring notebook can
# join/group on it. (Idempotent: won't duplicate the column on re-run.)
df_export = df_out.copy()
if "prompt_version" not in df_export.columns:
    df_export.insert(0, "prompt_version", PROMPT_VERSION)

# One row per prompt, full text preserved verbatim — this is your appendix evidence.
prompts_df = pd.DataFrame(
    [{"prompt_version": PROMPT_VERSION, "prompt_key": k, "prompt_text": v}
     for k, v in P.items()]
)

# Adding Summary sheet with counts and percentages of each mode, plus a total row for self-checking.
# Mode distribution — one row per mode, tagged with prompt_version for cross-run joins.
mode_summary = (
    df_out["mode"]
    .value_counts(dropna=False)
    .rename_axis("mode")
    .reset_index(name="count")
)
mode_summary["pct"] = (mode_summary["count"] / len(df_out) * 100).round(1)
mode_summary.insert(0, "prompt_version", PROMPT_VERSION)

# Total row so the sheet is self-checking against the run size.
mode_summary = pd.concat([
    mode_summary,
    pd.DataFrame([{"prompt_version": PROMPT_VERSION, "mode": "TOTAL",
                   "count": len(df_out), "pct": 100.0}]),
], ignore_index=True)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_export.to_excel(writer, sheet_name="method1_results", index=False)
    prompts_df.to_excel(writer, sheet_name="prompts", index=False)
    mode_summary.to_excel(writer, sheet_name="mode_summary", index=False)
    
    # Make the prompt sheet readable: wrap the long prompt_text column.
    from openpyxl.styles import Alignment
    ws = writer.sheets["prompts"]
    ws.column_dimensions["A"].width = 16   # prompt_version
    ws.column_dimensions["B"].width = 20   # prompt_key
    ws.column_dimensions["C"].width = 120  # prompt_text
    for row in ws.iter_rows(min_row=2, min_col=3, max_col=3):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical="top")
    
    mode_summary.to_excel(writer, sheet_name="mode_summary", index=False)

    ws_m = writer.sheets["mode_summary"]
    ws_m.column_dimensions["A"].width = 16   # prompt_version
    ws_m.column_dimensions["B"].width = 18   # mode
    ws_m.column_dimensions["C"].width = 10   # count
    ws_m.column_dimensions["D"].width = 8    # pct

print("Saved:", xlsx_path)
print("Sheets: method1_results, prompts | prompt_version:", PROMPT_VERSION)
print("Sheets: method1_results, prompts, mode_summary | prompt_version:", PROMPT_VERSION)

Saved: graph_rag_method1_results_prompt3_20260627_224820.xlsx
Sheets: method1_results, prompts | prompt_version: prompt3
Sheets: method1_results, prompts, mode_summary | prompt_version: prompt3


### Notes
- `mode` tells you per question whether the answer came from `graph`, `vector_fallback`, or `failed`.
- `id` is preserved for joining against your gold answers in the separate scoring notebook.
- `graph_rows`, `generated_cypher`, `retrieved_context`, `cypher_error`, `error` give you the
  provenance to diagnose failures.
